In [ ]:
import pandas
from sqlalchemy import text
from forecast.db.session import get_engine
import matplotlib.pyplot as plt


def calculate_hourly_means(df):
    return df.groupby(df.index.hour)["clean"].mean()

def is_data_in_months(df, *args):
    return df[df.index.month.isin(list(args))]

query = text("select ts_utc, mwh from demand_hourly where ba = \'AVA\' order by ts_utc asc")

data = pandas.read_sql(query, get_engine(), parse_dates = ["ts_utc"]).set_index("ts_utc")

#Data cleaning & interpolation
bad = ((data["mwh"] < 0) | (data["mwh"] > 5000))
data["clean"] = data["mwh"].mask(bad).interpolate(method = "time")
data["imputable"] = bad

data.index = data.index.tz_convert("America/Los_Angeles")

#Plot setup
window = slice("2024-01-03", "2024-01-08")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(data.loc[window].index, data.loc[window, "clean"], label="cleaned")
ax.plot(data.loc[window].index, data.loc[window, "mwh"], label="mwh", alpha=0.5)
ax.set_ylabel("MWh")
ax.legend()
ax.set_ylim(500, 3000)
plt.show()


hourly_means = calculate_hourly_means(data).rename_axis("Hour(PST)")
print("Hourly Means:")
print(hourly_means)
hourly_means.plot()

#Weekend calculations
is_weekend = data.index.day_of_week >= 5
weekend_and_weekday_means = data.groupby([is_weekend, data.index.hour])["clean"].mean().unstack(0).rename_axis("Hour(PST)")
print("Weekend and weekday means")
print(weekend_and_weekday_means)
weekend_and_weekday_means.plot()

#Seasonal calculations
winter = is_data_in_months(data, 1, 2, 12)
spring = is_data_in_months(data, 3, 4, 5)
summer = is_data_in_months(data, 6, 7, 8)
fall = is_data_in_months(data, 9, 10, 11)
seasonal_means = pandas.concat([calculate_hourly_means(winter), calculate_hourly_means(spring), calculate_hourly_means(summer), calculate_hourly_means(fall)], axis = 1, keys = ["winter", "spring", "summer", "fall"]).rename_axis("Hour(PST)")

print("Seasonal means:")
print(seasonal_means)
seasonal_means.plot()


#Naive error calculations
d =data.copy()
d["prev_24"] = d["clean"].shift(24)
d["prev_168"] = d["clean"].shift(168)

valid168 = d["prev_168"].notna() & ~d["imputable"] & ~(d["imputable"].shift(168).fillna(True).astype(bool))
valid24 = d["prev_24"].notna() & ~d["imputable"] & ~(d["imputable"].shift(24).fillna(True).astype(bool))
valid = valid24 & valid168

naive_err_24 = (d.loc[valid, "clean"] - d.loc[valid, "prev_24"]).abs()
naive_err_168 = (d.loc[valid, "clean"] - d.loc[valid, "prev_168"]).abs()
print("Naive Error Means:")
print(f"24 hour:{naive_err_24.mean()}")
print(f"168 hour: {naive_err_168.mean()}")
naive_mape_24 = (naive_err_24 / d.loc[valid, "clean"]).mean() * 100
naive_mape_168 = (naive_err_168 / d.loc[valid, "clean"]).mean() * 100
print("Naive Error Percentage Means:")
print(f"24 hour: {naive_mape_24}")
print(f"168 hour: {naive_mape_168}")

#Plots mean absolute errors by day
daily_24 = naive_err_24.resample("D").mean()
daily_168 = naive_err_168.resample("D").mean()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_24.index, daily_24, label="24h baseline", alpha=0.7)
ax.plot(daily_168.index, daily_168, label="168h baseline", alpha=0.7)
ax.set_ylabel("Mean absolute error (MWh)")
ax.legend()
plt.show()


In [2]:
import pandas
from sqlalchemy import text
import matplotlib.pyplot as plt
from forecast.db.session import get_engine

query = text("select ts_utc, temp_c from weather_hourly order by ts_utc asc")

data = pandas.read_sql(query, get_engine(), parse_dates = ["ts_utc"]).set_index("ts_utc")
print(data)

                           temp_c
ts_utc                           
2023-01-01 00:00:00+00:00     1.3
2023-01-01 01:00:00+00:00     1.0
2023-01-01 02:00:00+00:00    -0.5
2023-01-01 03:00:00+00:00    -3.0
2023-01-01 04:00:00+00:00    -3.0
...                           ...
2026-11-09 19:00:00+00:00     7.7
2026-11-09 20:00:00+00:00     9.5
2026-11-09 21:00:00+00:00    10.6
2026-11-09 22:00:00+00:00    11.2
2026-11-09 23:00:00+00:00    10.9

[33816 rows x 1 columns]
